**6.6 Standardization and Normalization**

Dataset: Pima Indians Diabetes `diabetes.csv`

**Goal:**
- Apply StandardScaler and MinMaxScaler (only one needed in actual workflow)
- Fit scalers on TRAIN only
- Transform both TRAIN and TEST

**1. Setup**

In [1]:
%pip install -q pandas numpy scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: '%.3f' % x)

**2. Load dataset**

In [3]:
df = pd.read_csv("diabetes.csv")
print("Shape:", df.shape)
df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.600,0.627,50,1
1,1,85,66,29,0,26.600,0.351,31,0
2,8,183,64,0,0,23.300,0.672,32,1
3,1,89,66,23,94,28.100,0.167,21,0
4,0,137,40,35,168,43.100,2.288,33,1


**3. Train test split**

In [4]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nShapes after split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# All columns are numeric in this dataset - do this manually if some columns are already encoded
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [col for col in X_train.columns if col not in num_cols]


Shapes after split:
X_train: (614, 8)
X_test : (154, 8)


In [5]:
print(num_cols)
print(cat_cols)

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
[]


**4. Standardization - StandardScaler**

**Standardization vs Normalization (example)**

- **Standardization (StandardScaler)**: centers features to mean 0 and unit variance. Fit on `X_train` then transform both `X_train` and `X_test` with the same scaler.
- **Normalization (MinMaxScaler)**: rescales features into the range [0, 1]. Also fit on `X_train` only.
- **Important:** always `fit` scalers on the training data (no leakage), then `transform` both train and test using the fitted scaler.

Below we demonstrate both scalers on the numeric columns of this dataset.

- mean ~ 0
- standard deviation ~ 1

**z = (x - mean) / std.dev**. #scale value

In [7]:
scaler = StandardScaler()

In [8]:
scaler.fit(X_train[num_cols])

StandardScaler()

In [9]:
X_train_std = X_train.copy()
X_test_std = X_test.copy()

In [10]:
X_train_std[num_cols] = scaler.transform(X_train[num_cols])
X_test_std[num_cols] = scaler.transform(X_test[num_cols])

In [11]:
print(X_train.describe().T)

                           count    mean     std    min    25%     50%  \
Pregnancies              614.000   3.819   3.314  0.000  1.000   3.000   
Glucose                  614.000 120.909  31.561  0.000 99.000 117.000   
BloodPressure            614.000  69.443  18.403  0.000 62.500  72.000   
SkinThickness            614.000  20.777  15.856  0.000  0.000  23.000   
Insulin                  614.000  78.666 107.737  0.000  0.000  40.500   
BMI                      614.000  31.973   7.861  0.000 27.500  32.300   
DiabetesPedigreeFunction 614.000   0.477   0.330  0.084  0.245   0.383   
Age                      614.000  33.366  11.833 21.000 24.000  29.000   

                             75%     max  
Pregnancies                6.000  17.000  
Glucose                  140.000 199.000  
BloodPressure             80.000 122.000  
SkinThickness             32.000  99.000  
Insulin                  130.000 744.000  
BMI                       36.500  67.100  
DiabetesPedigreeFunction   0.639

In [14]:
print("Standard Scaled summary for TRAIN:")
print(X_train_std.describe().T)

Standard Scaled summary for TRAIN:
                           count   mean   std    min    25%    50%   75%   max
Pregnancies              614.000 -0.000 1.001 -1.153 -0.851 -0.247 0.659 3.980
Glucose                  614.000  0.000 1.001 -3.834 -0.695 -0.124 0.605 2.476
BloodPressure            614.000 -0.000 1.001 -3.777 -0.378  0.139 0.574 2.858
SkinThickness            614.000 -0.000 1.001 -1.311 -1.311  0.140 0.708 4.937
Insulin                  614.000 -0.000 1.001 -0.731 -0.731 -0.355 0.477 6.181
BMI                      614.000  0.000 1.001 -4.070 -0.569  0.042 0.576 4.472
DiabetesPedigreeFunction 614.000 -0.000 1.001 -1.192 -0.704 -0.288 0.490 5.610
Age                      614.000 -0.000 1.001 -1.046 -0.792 -0.369 0.646 4.029


**5. Normalization - MinMaxScaler**

min value = 0

max value = 1

**x_scaled = (x - x_min) / (x_max - x_min)**

In [12]:
mm_scaler = MinMaxScaler()

mm_scaler.fit(X_train[num_cols])

X_train_mm = X_train.copy()
X_test_mm = X_test.copy()

X_train_mm[num_cols] = mm_scaler.transform(X_train[num_cols])
X_test_mm[num_cols] = mm_scaler.transform(X_test[num_cols])

In [13]:
print("MinMaxScaler summary for TRAIN:")
print(X_train_mm.describe().T)

MinMaxScaler summary for TRAIN:
                           count  mean   std   min   25%   50%   75%   max
Pregnancies              614.000 0.225 0.195 0.000 0.059 0.176 0.353 1.000
Glucose                  614.000 0.608 0.159 0.000 0.497 0.588 0.704 1.000
BloodPressure            614.000 0.569 0.151 0.000 0.512 0.590 0.656 1.000
SkinThickness            614.000 0.210 0.160 0.000 0.000 0.232 0.323 1.000
Insulin                  614.000 0.106 0.145 0.000 0.000 0.054 0.175 1.000
BMI                      614.000 0.477 0.117 0.000 0.410 0.481 0.544 1.000
DiabetesPedigreeFunction 614.000 0.175 0.147 0.000 0.072 0.133 0.247 1.000
Age                      614.000 0.206 0.197 0.000 0.050 0.133 0.333 1.000


In [15]:
X_train_mm.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,0.059,0.452,0.508,0.121,0.058,0.405,0.221,0.050
711,0.294,0.633,0.639,0.273,0.030,0.441,0.158,0.317
373,0.118,0.528,0.475,0.404,0.126,0.520,0.063,0.067
46,0.059,0.734,0.459,0.000,0.000,0.443,0.214,0.133
682,0.000,0.477,0.525,0.394,0.141,0.665,0.126,0.017


In [16]:
X_train_std.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.851,-0.980,-0.405,-0.554,-0.331,-0.608,0.311,-0.792
711,0.357,0.161,0.465,0.393,-0.526,-0.302,-0.116,0.561
373,-0.549,-0.504,-0.622,1.213,0.142,0.373,-0.765,-0.708
46,-0.851,0.796,-0.731,-1.311,-0.731,-0.289,0.262,-0.369
682,-1.153,-0.822,-0.296,1.150,0.245,1.607,-0.338,-0.961


**6. Save scaled versions**

In [17]:
train_std = X_train_std.copy()
train_std["Outcome"] = y_train.values

test_std = X_test_std.copy()
test_std["Outcome"] = y_test.values

train_mm = X_train_mm.copy()
train_mm["Outcome"] = y_train.values

test_mm = X_test_mm.copy()
test_mm["Outcome"] = y_test.values

train_std.to_csv("diabetes_train_standard_scaled.csv", index=False)
test_std.to_csv("diabetes_test_standard_scaled.csv", index=False)

train_mm.to_csv("diabetes_train_minmax_scaled.csv", index=False)
test_mm.to_csv("diabetes_test_minmax_scaled.csv", index=False)

print("\nSaved files:")
print(" - diabetes_train_standard_scaled.csv")
print(" - diabetes_test_standard_scaled.csv")
print(" - diabetes_train_minmax_scaled.csv")
print(" - diabetes_test_minmax_scaled.csv")


Saved files:
 - diabetes_train_standard_scaled.csv
 - diabetes_test_standard_scaled.csv
 - diabetes_train_minmax_scaled.csv
 - diabetes_test_minmax_scaled.csv
